# 1. Mount gdrive and import libs

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Literal
import matplotlib.pyplot as plt

from skimage.filters import gaussian, threshold_otsu
from skimage.segmentation import *
from skimage.color import rgb2gray, rgb2hsv
import skimage.io as skio
from skimage.feature import graycomatrix, graycoprops
import skimage.morphology as skmor
import skimage.measure as skmea
from skimage.morphology import (
    disk,
    binary_opening,
    binary_closing,
    remove_small_objects,
    remove_small_holes
)

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


# Unzip dataset zip file

In [12]:
!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"
!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"

Archive:  /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/027.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/027_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/032.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/032_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/034.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/034_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/071.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/071_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/076.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/076_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/082.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/082_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep

# Dataset class definition:
Chatgpt5.4 is used for designing, debugging and adding boundary condition structure of this dataset class.

In [ ]:
class Hep2Dataset(Dataset):
	def __init__(self, dataset_root, csv_path, transform=None):
		"""
		Initialize the dataset.

		Args:
			dataset_root (str or Path): Root folder of the image dataset.
			csv_path (str or Path): CSV file containing image file names and labels.
			transform (callable, optional): Transform to apply to each image.
		"""
		self.dataset_root = Path(dataset_root)
		self.csv_path = Path(csv_path)
		self.transform = transform

		# Read all samples once and store them as a unified list
		self.samples = self.read_label(self.csv_path)

	def read_label(self, csv_path):
		"""
		Read the CSV file and build a list of (image_path, label).

		Args:
			csv_path (Path): Path to the CSV label file.

		Returns:
			list: A list of tuples, where each tuple is (image_path, label).
		"""
		df = pd.read_csv(csv_path)

		samples = []
		for _, row in df.iterrows():
			img_name = row["file"]
			label = int(row["class"])  # Convert label to integer
			mask_path = self.dataset_root / f'{img_name}_Mask.png'
			img_path = self.dataset_root / f'{img_name}.png'
			samples.append((img_path, mask_path, label))

		return samples

	def __len__(self):
		"""
		Return the total number of samples in the dataset.
		"""
		return len(self.samples)

	def __getitem__(self, idx):
		"""
		Get one sample by index.

		Args:
			idx (int): Index of the sample.

		Returns:
			tuple: (image, mask, label)
		"""
		img_path, mask_path, label = self.samples[idx]
		img = skio.imread(img_path)
		mask = skio.imread(mask_path)

		if self.transform is not None:
			img = self.transform(img)

		return img, mask, label

# Shape feature extractor:
1.Chatgpt5.4 used for designing shape feature extraction class.

2.Here I used circularity, solidity and eccentricity beacuse we don't have metadata like what we have in DICOM format. So some features like perimeter and area, we cannnot get.

In [ ]:
class ShapeFeatureExtractor:
    def __init__(self, image, label, mask):
        """
        Initialize the shape feature extractor.

        Args:
            image (ndarray): Input image.
            label (int): Corresponding label.
			mask (ndarray): Corresponding mask.
        """
        self.image = image
        self.label = label
        self.mask = mask

        self.circularity = []
        self.solidity = []
        self.eccentricity = []

        self.shape_feature = {}

    def _get_largest_region(self, mask):
        """
        Get the largest connected component from a mask.

        Args:
            mask (ndarray): Input mask.

        Returns:
            regionprops object or None: Largest region if exists, else None.
        """
        mask = mask > 0
        labeled_mask = skmea.label(mask)

        regions = skmea.regionprops(labeled_mask)
        if len(regions) == 0:
            return None

        largest_region = max(regions, key=lambda x: x.area)
        return largest_region

    def get_solidity(self, mask):
        """
        Calculate solidity of the largest connected component.

        Args:
            mask (ndarray): Input mask.

        Returns:
            float: Solidity value. Returns 0.0 if no valid region exists.
        """
        largest_region = self._get_largest_region(mask)
        if largest_region is None:
            return 0.0

        solidity = largest_region.solidity
        return float(solidity)

    def get_eccentricity(self, mask):
        """
        Calculate eccentricity of the largest connected component.

        Args:
            mask (ndarray): Input mask.

        Returns:
            float: Eccentricity value. Returns 0.0 if no valid region exists.
        """
        largest_region = self._get_largest_region(mask)
        if largest_region is None:
            return 0.0

        eccentricity = largest_region.eccentricity
        return float(eccentricity)

    def get_circularity(self, mask):
        """
        Calculate circularity of the largest connected component.

        Formula:
            circularity = 4 * pi * area / perimeter^2

        Args:
            mask (ndarray): Input mask.

        Returns:
            float: Circularity value. Returns 0.0 if no valid region exists
                   or perimeter is zero.
        """
        largest_region = self._get_largest_region(mask)
        if largest_region is None:
            return 0.0

        area = largest_region.area
        perimeter = largest_region.perimeter

        if perimeter == 0:
            return 0.0

        circularity = 4.0 * np.pi * area / (perimeter ** 2)
        return float(circularity)

    def extract_shape_feature(self, mask):
        """
        Extract all shape features from a single mask.

        Args:
            mask (ndarray): Input mask.

        Returns:
            dict: Dictionary containing shape features.
        """
        single_feature = {
            "circularity": self.get_circularity(mask),
            "solidity": self.get_solidity(mask),
            "eccentricity": self.get_eccentricity(mask),
            "label": self.label
        }
        return single_feature

# GLCM feature exatractor:

In [ ]:
class GlcmFeatureExtractor:
	def __init__(self, image, label, mask):
		"""
		Initialize the GLCM feature extractor.

		Args:
			image (ndarray): Input image.
			label (int): Corresponding label.
			mask (ndarray): Corresponding mask.
		"""
		self.image = image
		self.label = label
		self.mask = mask

		# glcm configuration
		self.distance = 1
		self.angle = 0
		self.levels = 256
		self.symmetric = False
		self.normed = False

		# glcm features
		self.contrast = []
		self.dissimilarity = []
		self.homogeneity = []
		self.energy = []
		self.correlation = []

		self.glcm_feature = {}

	def _get_largest_region(self, mask):
		"""
		Get the largest connected component from a mask.

		Args:
			mask (ndarray): Input mask.
		Returns:
			regionprops object or None: Largest region if exists, else None.
		"""
		mask = mask > 0
		labeled_mask = skmea.label(mask)
		regions = skmea.regionprops(labeled_mask)
		if len(regions) == 0:
			return None
		largest_region = max(regions, key = lambda x : x.area)
		return largest_region
	
	def get_masked_gray_image(self, image, mask):
		"""
		Get the gray-level image with the background masked out.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			ndarray: Gray-level image with background masked out.
		"""
		background_mask  = ~(mask > 0)
		gray_image = rgb2gray(image)

		# only keep foreground region
		masked_image = gray_image.copy()
		masked_image[background_mask] = 0
		return masked_image
	
	def _get_glcm_features(self, masked_image):
		# get GLCM matrix
		glcm = graycomatrix(
					masked_image.astype(np.uint8), 
					distances=[self.distance], 
					angles=[self.angle], 
					levels=self.levels, 
					symmetric=self.symmetric, 
					normed=self.normed
					)
		# extract GLCM features
		glcm_contrast = graycoprops(glcm, prop='contrast')[0, 0]
		glcm_dissimilarity = graycoprops(glcm, prop='dissimilarity')[0, 0]
		glcm_homogeneity = graycoprops(glcm, prop='homogeneity')[0, 0]
		glcm_energy = graycoprops(glcm, prop='energy')[0, 0]
		glcm_correlation = graycoprops(glcm, prop='correlation')[0, 0]

		return glcm_contrast, glcm_dissimilarity, glcm_homogeneity, glcm_energy, glcm_correlation
	
	def extract_glcm_feature(self, masked_images: list):
		"""
		Extract GLCM features from a masked image.

		Args:
			masked_images (list): List of input masked images.
		Returns:
			list: List of dictionaries containing GLCM features for each masked image.
		"""
		glcm_features = []
		for masked_image in masked_images:
			glcm_contrast, glcm_dissimilarity, glcm_homogeneity, glcm_energy, glcm_correlation = self._get_glcm_features(masked_image)
			feature_dict = {
				"contrast": glcm_contrast,
				"dissimilarity": glcm_dissimilarity,
				"homogeneity": glcm_homogeneity,
				"energy": glcm_energy,
				"correlation": glcm_correlation,
				"label": self.label
			}
			glcm_features.append(feature_dict)
		return glcm_features

In [ ]:
class ColorFeatureExtractor:
	def __init__(self, image, label, mask):
		"""
		Initialize the color feature extractor.

		Args:
			image (ndarray): Input image.
			label (int): Corresponding label.
			mask (ndarray): Corresponding mask.
		"""
		self.image = image
		self.label = label
		self.mask = mask

		self.mean_intensity = []
		self.std_intensity = []
	
	def _get_largest_region(self, mask):
		"""
		Get the largest connected component from a mask.

		Args:
			mask (ndarray): Input mask.
		Returns:
			regionprops object or None: Largest region if exists, else None.
		"""
		mask = mask > 0
		labeled_mask = skmea.label(mask)
		regions = skmea.regionprops(labeled_mask)
		if len(regions) == 0:
			return None
		largest_region = max(regions, key = lambda x : x.area)
		return largest_region
	
	def get_masked_color_image(self, image, mask):
		"""
		Get the color image with the background masked out.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			ndarray: Color image with background masked out.
		"""
		background_mask  = ~(mask > 0)
		masked_image = image.copy()
		masked_image[background_mask] = 0
		return masked_image
	
	def get_color_histogram(self, masked_image):
		"""
		Calculate the mean and standard deviation of pixel intensities in the masked image.

		Args:
			masked_image (ndarray): Input masked image.
		Returns:
			tuple: (mean_intensity, std_intensity) where each is a list of mean and std for each color channel.
		"""
		hists = {}
		channels = ['R', 'G', 'B']
		for channel in range(masked_image.shape[2]):
			channel_data = masked_image[:, :, channel]
			channel_histogram, bin_edges = np.histogram(channel_data[channel_data > 0], bins=256, range=(0, 255))
			hists[channels[channel]] = channel_histogram
		return hists, bin_edges

In [ ]:
# These are for exporting the notebook as a PDF later if needed
!apt-get update
!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!jupyter nbconvert --to pdf /content/gdrive/MyDrive/Colab_Notebooks/BMET5933/WEEK_8/Week_8_9_Last_name_Code_file_ipynb_draft_version.ipynb

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
texlive-fonts-recommended is already 